In [7]:
from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import Element, Image, FigureCaption, Table, Text


def parse(path: str) -> list[Element]:
    elements = partition_pdf(
        filename=path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image", "Figure", "Table"],
        extract_image_block_to_payload=True,
        chunking_strategy=None,
    )
    return elements




In [8]:
path = "/home/arjav.singh/Projects/hr-rag/documents/2312.10997v5.pdf"
elements = parse(path)

In [28]:
# Unique data types in the document
print(set([type(e) for e in elements]))

{<class 'unstructured.documents.elements.Text'>, <class 'unstructured.documents.elements.Header'>, <class 'unstructured.documents.elements.Title'>, <class 'unstructured.documents.elements.NarrativeText'>, <class 'unstructured.documents.elements.Table'>, <class 'unstructured.documents.elements.Image'>, <class 'unstructured.documents.elements.ListItem'>, <class 'unstructured.documents.elements.FigureCaption'>}


In [29]:
for idx, element in enumerate(elements):
    if isinstance(element, Image):
        print(f"Image found at index {idx}")
    elif isinstance(element, Table):
        print(f"Table found at index {idx}")
    

Image found at index 27
Image found at index 40
Image found at index 54
Table found at index 79
Image found at index 299
Image found at index 363
Table found at index 400
Table found at index 441
Table found at index 457
Image found at index 470


## Image Parsing

In [20]:
def collect_image_data(elements: list[Element]) -> list[Image]:
    images = []
    for idx, element in enumerate(elements):
        if isinstance(element, Image):
            
            if idx + 1 < len(elements) and isinstance(elements[idx + 1], FigureCaption):
                caption = elements[idx + 1].text
            else:
                caption = None
                
            
            props = {
                'index': idx,
                'caption': caption if caption else "No caption",
                'image_text': element.text if element.text else "No text",
                'image_base64': element.to_dict().get('metadata', {}).get('image_base64', "No base64 data")
            }
            
            images.append(props)
                
    return images

In [21]:
images = collect_image_data(elements)

## Table Parsing

In [25]:
def collect_table_data(elements: list[Element]) -> list[Table]:
    tables = []
    for idx, element in enumerate(elements):
        if isinstance(element, Table):
            
            props = {
                'index': idx,
                'table_text': element.text if element.text else "No text",
                'text_as_html': element.to_dict().get("metadata", {}).get('text_as_html', "No html data")
            }
            
            tables.append(props)
                
    return tables

In [30]:
tables = collect_table_data(elements)

In [46]:
tables

[{'index': 79,
  'table_text': 'Retrieval Source Retrieval Data Type Wikipedia Text FactoidWiki Text Dataset-base Text Dataset-base Text Dataset-base Text Dataset-base Text Search Engine,Wikipedia Text Wikipedia Text Wikipedia Text Dataset-base Text Synthesized dataset Text Dataset-base Text Dataset-base Text Dataset-base Text Dataset-base Text Dataset-base Text Synthesized dataset Text Wikipedia, Common Crawl Text Wikipedia Text Pre-training Corpus Text Pre-training corpus Text Search Engine Text Dataset-base Text BEIR Text MSMARCO,Wikipedia Text Common Crawl,Wikipedia Text Wikipedia Text Dataset-base Text Wikipedia Text Wikipedia Text Wikipedia Text Wikipedia Text Wikipedia Text Arxiv,Online Database,PubMed Text FactoidWiki Text Search Engine,Wikipedia Text Wikipedia Text Search Engine,Wikipedia Text Dataset-base,Wikipedia Text Wikipedia Text Dataset-base Text Wikipedia Text Wikipedia Text Wikipedia Text Dataset-base Text LLMs Text Pile,Wikipedia Text Dataset-base Text C4 Text Arxiv 

## Captioning

In [ ]:
import ollama

def table_captioning(table_data):
    errors = []
    processed_table_data = []

    try:
        response = ollama.chat(
            model="deepseek-r1:latest",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert at understanding structured data."
                        " Analyze tables and summarize structure, key values,"
                        " relationships, and insights."
                    )
                },
                {
                    "role": "user",
                    "content": (
                        "Analyze the following table and provide a detailed summary.\n\n"
                        "Return only the summary.\n\n"
                        f"Table (HTML):\n{table_data["text_as_html"]}"
                    )
                }
            ],
            options={"temperature": 0.2}
        )

        table_data["content"] = response["message"]["content"]

    except Exception as e:
        errors.append({
            "error": str(e),
            "error_message": "Error generating description with Ollama."
        })

    processed_table_data.append(table_data)
    return processed_table_data, errors

In [ ]:
processed_table_data, error = table_captioning(tables[0])

In [52]:
error

[{'error': "'dict' object has no attribute 'text_as_html'",
  'error_message': 'Error generating description with Ollama.'}]

In [51]:
tables[0]

{'index': 79,
 'table_text': 'Retrieval Source Retrieval Data Type Wikipedia Text FactoidWiki Text Dataset-base Text Dataset-base Text Dataset-base Text Dataset-base Text Search Engine,Wikipedia Text Wikipedia Text Wikipedia Text Dataset-base Text Synthesized dataset Text Dataset-base Text Dataset-base Text Dataset-base Text Dataset-base Text Dataset-base Text Synthesized dataset Text Wikipedia, Common Crawl Text Wikipedia Text Pre-training Corpus Text Pre-training corpus Text Search Engine Text Dataset-base Text BEIR Text MSMARCO,Wikipedia Text Common Crawl,Wikipedia Text Wikipedia Text Dataset-base Text Wikipedia Text Wikipedia Text Wikipedia Text Wikipedia Text Wikipedia Text Arxiv,Online Database,PubMed Text FactoidWiki Text Search Engine,Wikipedia Text Wikipedia Text Search Engine,Wikipedia Text Dataset-base,Wikipedia Text Wikipedia Text Dataset-base Text Wikipedia Text Wikipedia Text Wikipedia Text Dataset-base Text LLMs Text Pile,Wikipedia Text Dataset-base Text C4 Text Arxiv Te